In [ ]:
# Setup — mirror img2text_worker.ipynb; batch mode needs no Kaggle Secrets at all
!apt-get -qq install -y poppler-utils
!pip install -q paddlepaddle-gpu paddleocr httpx opencv-python-headless rapidfuzz pypdf pillow numpy
!pdftoppm -v

In [ ]:
# Locate input — first jpg/jpeg/png/webp/pdf in the img2text-input dataset mount
import glob
import os

EXTS = {".jpg", ".jpeg", ".png", ".webp", ".pdf"}
CANDIDATES = sorted(
    p for p in glob.glob("/kaggle/input/img2text-input/*")
    if os.path.splitext(p)[1].lower() in EXTS
)
if not CANDIDATES:
    print("ERROR: no input file (jpg/jpeg/png/webp/pdf) in /kaggle/input/img2text-input/")
    print("hint: push a file first — run_one.py stages it into the img2text-input dataset")
    raise SystemExit(1)
INPUT_PATH = CANDIDATES[0]
print("input:", INPUT_PATH)

In [ ]:
# Process — reuse worker/pipeline.py helpers; no OCR logic duplicated here.
# Reused: default_pdftoppm (PDF rasterize), precheck (verdict),
# build_page (adapter+verdict per page), build_result (top shape) — same
# per-page flow as run_job, minus the server loop/heartbeat.
import io
import sys
import time

import numpy as np
from PIL import Image

sys.path.insert(0, "/kaggle/input/img2text-worker-files/worker")
try:
    from pipeline import build_page, build_result, default_pdftoppm, precheck
except ImportError:
    # ponytail: layout fallback for datasets pushed by push.py (files at dataset root)
    sys.path.insert(0, "/kaggle/input/img2text-worker-files")
    from pipeline import build_page, build_result, default_pdftoppm, precheck

from paddleocr import PaddleOCRVL

PIPELINE = PaddleOCRVL(
    pipeline_version="v1.5",
    vl_rec_model_name="PaddleOCR-VL-1.5-0.9B",
    layout_detection_model_name="PP-DocLayoutV3",
)


def process_one(path, PIPELINE):
    """Single input file -> build_result dict (spec §4)."""
    with open(path, "rb") as f:
        data = f.read()
    if path.lower().endswith(".pdf"):
        raster = default_pdftoppm(data, "pdf")
    else:
        im = Image.open(io.BytesIO(data))
        raster = [(data, im.size[0], im.size[1])]
    pages = []
    gpu_seconds = 0.0
    for i, png, w, h in enumerate(raster, 1):
        arr = np.array(Image.open(io.BytesIO(png)))
        pc = precheck(arr)
        if pc.blurry and pc.too_small:
            pages.append(build_page({"width": w, "height": h}, i, inference_ran=False))
            continue
        t0 = time.monotonic()
        out = PIPELINE.predict(arr)
        gpu_seconds += time.monotonic() - t0
        out.json.setdefault("width", w)
        out.json.setdefault("height", h)
        pages.append(build_page(out.json, i, precheck=pc, inference_ran=True))
    return build_result(pages, gpu_seconds)

In [ ]:
# Output — /kaggle/outputs is what kernels_output pulls back to the client
import json
import os

result = process_one(INPUT_PATH, PIPELINE)
with open("/kaggle/outputs/result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
if result.get("markdown"):
    with open("/kaggle/outputs/result.md", "w", encoding="utf-8") as f:
        f.write(result["markdown"])
print("files:", sorted(os.listdir("/kaggle/outputs")))
print("done")